# Bronze Generic Pipeline - Config Driven

**Metadata-Driven Ingestion Framework**

This notebook reads ANY source defined in `pipeline_config.json` and:
* Loads data from configured source
* Applies DQ rules from config
* Writes to Bronze tables with audit metadata
* Quarantines failed records

**Parameters:**
* `source_name` - Source identifier from config (e.g., 'customers', 'transactions')
* `environment` - Environment (dev/prod)
* `run_mode` - 'full' or 'incremental'

In [0]:
# Notebook parameters (can be overridden by job)
dbutils.widgets.text("source_name", "customers", "Source Name")
dbutils.widgets.dropdown("environment", "dev", ["dev", "prod"], "Environment")
dbutils.widgets.dropdown("run_mode", "full", ["full", "incremental"], "Run Mode")

source_name = dbutils.widgets.get("source_name")
environment = dbutils.widgets.get("environment")
run_mode = dbutils.widgets.get("run_mode")

print(f"🎯 Bronze Ingestion Parameters:")
print(f"   Source: {source_name}")
print(f"   Environment: {environment}")
print(f"   Mode: {run_mode}")

In [0]:
import json
from pyspark.sql import functions as F
from datetime import datetime

# Load pipeline configuration
config_path = "/Workspace/Users/jayarampogakula@gmail.com/lakeforge/config/pipeline_config.json"
with open(config_path, 'r') as f:
    config = json.load(f)

# Extract environment config
env_config = config['environments'][environment]
catalog = env_config['catalog']
bronze_schema = env_config['bronze_schema']

# Extract source config
if source_name not in config['sources']:
    raise ValueError(f"Source '{source_name}' not found in config. Available: {list(config['sources'].keys())}")

source_config = config['sources'][source_name]
dq_rules = config['dq_rules'].get(source_name, [])

print(f"✅ Config loaded successfully")
print(f"   Target: {catalog}.{bronze_schema}.{source_config['bronze_table']}")
print(f"   DQ Rules: {len(dq_rules)} configured")

In [0]:
# Create catalog and schema if needed
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{bronze_schema}")

print(f"✅ Schema {catalog}.{bronze_schema} ready")

In [0]:
# Load data based on source type
file_path = source_config['file_path']
file_format = source_config['file_format']
options = source_config.get('options', {})

if source_config['source_type'] == 'csv':
    df = spark.read.format(file_format) \
        .options(**options) \
        .load(file_path)
else:
    raise ValueError(f"Unsupported source type: {source_config['source_type']}")

# Add audit columns
df = df.withColumn("_source_file", F.lit(file_path.split('/')[-1])) \
       .withColumn("_ingestion_timestamp", F.lit(datetime.now())) \
       .withColumn("_environment", F.lit(environment))

print(f"✅ Loaded {df.count()} rows from {file_path}")
print(f"   Schema: {df.columns}")
display(df.limit(5))

In [0]:
# Apply DQ rules from config
from pyspark.sql.functions import col, count, when, isnan, isnull

dq_results = []
failed_rows = None

for rule in dq_rules:
    rule_id = rule['rule_id']
    rule_name = rule['rule_name']
    validation_type = rule['validation_type']
    severity = rule['severity']
    action = rule['action']
    
    if validation_type == 'not_null':
        column = rule['column']
        null_count = df.filter(col(column).isNull()).count()
        passed = null_count == 0
        
        dq_results.append({
            'rule_id': rule_id,
            'rule_name': rule_name,
            'passed': passed,
            'failed_count': null_count,
            'severity': severity
        })
        
        if not passed and action == 'quarantine':
            if failed_rows is None:
                failed_rows = df.filter(col(column).isNull()).withColumn("_dq_failure_reason", F.lit(rule_name))
            else:
                failed_rows = failed_rows.union(
                    df.filter(col(column).isNull()).withColumn("_dq_failure_reason", F.lit(rule_name))
                )
    
    elif validation_type == 'unique':
        column = rule['column']
        duplicate_count = df.groupBy(column).count().filter(col('count') > 1).count()
        passed = duplicate_count == 0
        
        dq_results.append({
            'rule_id': rule_id,
            'rule_name': rule_name,
            'passed': passed,
            'failed_count': duplicate_count,
            'severity': severity
        })
        
        if not passed and action == 'quarantine':
            duplicates = df.groupBy(column).count().filter(col('count') > 1).select(column)
            dup_rows = df.join(duplicates, on=column, how='inner').withColumn("_dq_failure_reason", F.lit(rule_name))
            if failed_rows is None:
                failed_rows = dup_rows
            else:
                failed_rows = failed_rows.union(dup_rows)
    
    elif validation_type == 'null_rate_threshold':
        column = rule['column']
        threshold = rule['threshold']
        total_count = df.count()
        null_count = df.filter(col(column).isNull()).count()
        null_rate = null_count / total_count if total_count > 0 else 0
        passed = null_rate <= threshold
        
        dq_results.append({
            'rule_id': rule_id,
            'rule_name': rule_name,
            'passed': passed,
            'failed_count': f"{null_rate*100:.1f}% (threshold: {threshold*100}%)",
            'severity': severity
        })

# Display DQ results
print("=" * 80)
print("DATA QUALITY VALIDATION RESULTS")
print("=" * 80)
for result in dq_results:
    status = "✅ PASS" if result['passed'] else "❌ FAIL"
    print(f"{status} [{result['severity']}] {result['rule_name']}: {result['failed_count']} failures")
print("=" * 80)

In [0]:
# Separate clean records from quarantined
if failed_rows is not None:
    failed_ids = failed_rows.select(*source_config['business_key']).distinct()
    passed_df = df.join(failed_ids, on=source_config['business_key'], how='left_anti')
    print(f"✅ Clean records: {passed_df.count()}")
    print(f"❌ Quarantined records: {failed_rows.count()}")
else:
    passed_df = df
    failed_rows = df.limit(0)  # Empty DataFrame
    print(f"✅ All {passed_df.count()} records passed DQ validation")

In [0]:
# Write clean records to bronze table
target_table = f"{catalog}.{bronze_schema}.{source_config['bronze_table']}"

if source_config['merge_strategy'] == 'upsert':
    # Use merge for upsert
    passed_df.createOrReplaceTempView("source_data")
    
    merge_keys = source_config['business_key']
    merge_condition = " AND ".join([f"target.{k} = source.{k}" for k in merge_keys])
    
    spark.sql(f"""
        MERGE INTO {target_table} AS target
        USING source_data AS source
        ON {merge_condition}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"✅ Merged {passed_df.count()} records into {target_table}")
    
elif source_config['merge_strategy'] == 'append':
    # Append strategy
    passed_df.write.format("delta") \
        .mode("append") \
        .saveAsTable(target_table)
    print(f"✅ Appended {passed_df.count()} records to {target_table}")

else:
    # Overwrite strategy
    passed_df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_table)
    print(f"✅ Overwrote {target_table} with {passed_df.count()} records")

# Verify
result_count = spark.table(target_table).count()
print(f"✅ Verification: {result_count} total records in {target_table}")

In [0]:
# Write quarantined records to separate table
if failed_rows.count() > 0:
    quarantine_table = f"{target_table}_quarantine"
    
    failed_rows.write.format("delta") \
        .mode("append") \
        .saveAsTable(quarantine_table)
    
    print(f"⚠️  {failed_rows.count()} records quarantined in {quarantine_table}")
else:
    print("✅ No records quarantined")

In [0]:
print("=" * 80)
print("BRONZE INGESTION COMPLETE")
print("=" * 80)
print(f"Source: {source_name}")
print(f"Target: {target_table}")
print(f"Total Loaded: {df.count()}")
print(f"Passed: {passed_df.count()}")
print(f"Quarantined: {failed_rows.count()}")
print(f"Success Rate: {passed_df.count()/df.count()*100:.1f}%")
print("=" * 80)